# Lab 10: Embedding Space Attacks — Adversarial Examples for Text
## Manipulating How AI "Sees" Language

**Workshop:** Attacking, Defending & Leveraging Agentic AI  
**Authors:** Derek Banks and Dr. Brian Fehrman  
**Time:** 35 minutes  
**Platform:** Google Colab (T4 recommended)  
**Theme:** Attack  

---

### What You Will Learn
- How text embeddings work and what the embedding space looks like
- How to visualize embedding clusters (malicious vs. benign content)
- How adversarial text manipulation can move samples between clusters
- How embedding-based classifiers and detectors can be fooled
- The direct connection to vector database security and RAG poisoning

### Prerequisites
- Basic understanding of machine learning concepts
- Familiarity with Python and NumPy
- A Google account for Colab access

### Threat Model Connection
This lab demonstrates how **embedding-based security controls** — the backbone of  
vector database filtering, semantic similarity search, and RAG guardrails — can be  
systematically defeated by an attacker who understands the geometry of the embedding space.  

Every time you hear "we use AI to detect malicious content," there is an adversarial  
blind spot. This lab teaches you to find it.

### Key References
- Goodfellow et al. (2015): "Explaining and Harnessing Adversarial Examples"
- Morris et al. (2020): "TextAttack: A Framework for Adversarial Attacks on NLP"
- Song et al. (2020): "Adversarial Semantic Collisions"
- Workshop Slide 78: Vector DB Security Architecture

In [ ]:
# ============================================================
# Cell 2: Environment Setup
# ============================================================
# Install all required packages for embedding space analysis
# and adversarial attack generation. sentence-transformers
# provides the embedding model; umap-learn provides
# dimensionality reduction for visualization.
# ============================================================

!pip install -q sentence-transformers scikit-learn matplotlib numpy umap-learn

import sys
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version: {sys.version}")

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("No GPU detected - CPU mode (will be slower but functional)")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib import cm
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics.pairwise import cosine_similarity
import umap

print("\nAll packages imported successfully.")
print("=" * 60)

---
## Part 1: Background — What Are Embeddings?

### Text as Geometry

Modern NLP systems do not process text as strings. They convert text into  
**dense numerical vectors** called **embeddings** — points in a high-dimensional  
space (typically 384 to 4096 dimensions).

**Key Properties:**
- Semantically similar texts are **close together** in embedding space
- Dissimilar texts are **far apart**
- The distance metric (cosine similarity) defines "meaning" for the model

### Why This Matters for Security

Many AI security systems rely on embeddings:
- **Vector databases** (Pinecone, Weaviate, ChromaDB) use embeddings for RAG retrieval
- **Content classifiers** embed text and classify by proximity to known categories
- **Semantic firewalls** block queries that are "too similar" to known attack patterns

**The vulnerability:** If an attacker can manipulate text so that its embedding  
moves from one cluster to another, they can bypass any embedding-based defense.

```
   Benign Cluster          Malicious Cluster
   +-------------+         +-------------+
   |  o  o       |         |  x  x       |
   |    o  o     |  <---   |    x  x     |
   |  o    o     |  ATTACK |  x    x     |
   +-------------+  MOVES  +-------------+
                    POINT
```

In this lab, we will:
1. Build the embedding space
2. Train a classifier on it
3. Systematically attack it with four different techniques
4. Measure which attacks succeed and which defenses hold

---
## Part 2: Building the Text Corpus

We create three categories of text that represent what a security  
system might encounter:

| Category | Color | Count | Description |
|----------|-------|-------|-------------|
| **Benign** | Green | 20 | Normal business communication |
| **Malicious** | Red | 20 | Attack-related content |
| **Ambiguous** | Yellow | 10 | Legitimate security work |

In [ ]:
# ============================================================
# Cell 4: Define the Text Corpus
# ============================================================
# 50 carefully crafted texts across three categories.
# These simulate real-world content that an embedding-based
# security classifier would need to distinguish.
# ============================================================

# --- BENIGN TEXTS (20) ---
benign_texts = [
    "Please review the Q3 financial report and send your feedback by Friday.",
    "The team meeting has been rescheduled to 2pm tomorrow in Conference Room B.",
    "I've attached the updated project timeline for your review and approval.",
    "Can you confirm the delivery schedule for the new office furniture order?",
    "Great work on the client presentation yesterday. The stakeholders were impressed.",
    "Reminder: annual performance reviews are due by end of month.",
    "The new employee onboarding documentation has been uploaded to SharePoint.",
    "Please submit your travel expense reports before the quarterly deadline.",
    "We need to update the customer database with the latest contact information.",
    "The marketing team has finalized the campaign strategy for next quarter.",
    "Server maintenance is scheduled for Saturday night from 10pm to 2am.",
    "The software license renewal invoice has been forwarded to procurement.",
    "Please complete the mandatory compliance training module by next Wednesday.",
    "The cafeteria menu for next week has been posted on the intranet.",
    "I'll be out of office next Monday and Tuesday for a conference in Denver.",
    "The quarterly sales figures show a 12% increase over the previous quarter.",
    "Can you set up a Zoom call with the vendor to discuss the contract terms?",
    "The IT department has resolved the email synchronization issue reported yesterday.",
    "New parking garage access cards will be distributed at the front desk.",
    "The annual company picnic is scheduled for July 15th at Riverside Park."
]

# --- MALICIOUS TEXTS (20) ---
malicious_texts = [
    "Use the SQL injection payload to extract the user credentials database.",
    "Deploy the ransomware payload once you have lateral movement across the domain.",
    "The phishing email template should mimic the CEO's writing style to bypass filters.",
    "Exfiltrate the customer PII data through the DNS tunneling channel.",
    "Escalate privileges using the unpatched kernel vulnerability CVE-2024-1234.",
    "Install the keylogger on the target workstation to capture admin credentials.",
    "The backdoor trojan should persist through system reboots using the registry.",
    "Craft a spear phishing message targeting the CFO with a fake invoice attachment.",
    "Use the zero-day exploit to gain remote code execution on the web server.",
    "Set up command and control infrastructure using compromised cloud instances.",
    "Brute force the SSH credentials using the leaked password list from the dark web.",
    "The malware dropper should evade antivirus detection using polymorphic code.",
    "Pivot from the compromised DMZ host to the internal Active Directory server.",
    "Create a watering hole attack on the industry forum the target employees visit.",
    "Use social engineering to trick the help desk into resetting the admin password.",
    "Establish persistence by adding a scheduled task that downloads the payload nightly.",
    "The credential harvesting page should be an exact clone of the corporate login portal.",
    "Intercept the authentication tokens using a man-in-the-middle proxy on the network.",
    "Encrypt all shared drives and demand payment in cryptocurrency for the decryption key.",
    "Deploy the rootkit to hide our processes and network connections from security tools."
]

# --- AMBIGUOUS TEXTS (10) ---
# Legitimate security research and penetration testing
ambiguous_texts = [
    "The penetration test report identified three critical vulnerabilities in the web application.",
    "We need to test the firewall rules by attempting to bypass them with crafted packets.",
    "The red team exercise revealed that social engineering was the most effective attack vector.",
    "Document the SQL injection findings and recommend remediation steps for the development team.",
    "The vulnerability assessment scan discovered an unpatched remote code execution flaw.",
    "Our security researchers analyzed the malware sample to understand its command and control protocol.",
    "The incident response team traced the breach to a compromised vendor VPN credential.",
    "We should simulate a phishing campaign to measure employee security awareness training effectiveness.",
    "The bug bounty program received a report about a privilege escalation vulnerability in the API.",
    "Review the MITRE ATT&CK framework mapping for our threat model and update detection rules."
]

# Combine all texts with labels
all_texts = benign_texts + malicious_texts + ambiguous_texts
labels = [0] * len(benign_texts) + [1] * len(malicious_texts) + [2] * len(ambiguous_texts)
label_names = {0: "Benign", 1: "Malicious", 2: "Ambiguous"}
label_colors = {0: "#2ecc71", 1: "#e74c3c", 2: "#f39c12"}

print(f"Corpus size: {len(all_texts)} texts")
print(f"  Benign:    {len(benign_texts)}")
print(f"  Malicious: {len(malicious_texts)}")
print(f"  Ambiguous: {len(ambiguous_texts)}")
print()
print("Sample benign:    ", benign_texts[0][:80], "...")
print("Sample malicious: ", malicious_texts[0][:80], "...")
print("Sample ambiguous: ", ambiguous_texts[0][:80], "...")

---
## Part 3: Generating Embeddings

We use **all-MiniLM-L6-v2**, a lightweight but effective sentence transformer  
that produces 384-dimensional embeddings. This model is commonly used in  
production vector databases and semantic search systems.

**Model Details:**
- Architecture: MiniLM (distilled BERT)
- Embedding dimensions: 384
- Training: 1B+ sentence pairs
- Speed: ~14,000 sentences/second on GPU

In [ ]:
# ============================================================
# Cell 6: Generate Embeddings
# ============================================================
# Load the sentence transformer model and encode all texts
# into 384-dimensional embedding vectors. These vectors are
# what security classifiers and vector databases actually
# operate on.
# ============================================================

print("Loading sentence transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
print()

# Generate embeddings for all texts
print("Generating embeddings for all texts...")
embeddings = model.encode(all_texts, show_progress_bar=True, convert_to_numpy=True)

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"  {embeddings.shape[0]} texts x {embeddings.shape[1]} dimensions")
print()

# Show a sample embedding vector
print("Sample embedding (first benign text, first 20 dims):")
print(np.array2string(embeddings[0][:20], precision=4, separator=', '))
print(f"  ... ({embeddings.shape[1] - 20} more dimensions)")
print()

# Show embedding statistics
print("Embedding Statistics:")
print(f"  Mean magnitude: {np.mean(np.linalg.norm(embeddings, axis=1)):.4f}")
print(f"  Std magnitude:  {np.std(np.linalg.norm(embeddings, axis=1)):.4f}")
print(f"  Min value:      {embeddings.min():.4f}")
print(f"  Max value:      {embeddings.max():.4f}")

In [ ]:
# ============================================================
# Cell 7: Examine Pairwise Similarities
# ============================================================
# Before visualizing, let's look at the raw cosine similarity
# between categories. This tells us how separable the
# categories are in the original 384-D space.
# ============================================================

benign_emb = embeddings[:len(benign_texts)]
malicious_emb = embeddings[len(benign_texts):len(benign_texts)+len(malicious_texts)]
ambiguous_emb = embeddings[len(benign_texts)+len(malicious_texts):]

# Compute centroids
benign_centroid = benign_emb.mean(axis=0)
malicious_centroid = malicious_emb.mean(axis=0)
ambiguous_centroid = ambiguous_emb.mean(axis=0)

# Intra-cluster similarities (how tight are the clusters?)
benign_intra = cosine_similarity(benign_emb).mean()
malicious_intra = cosine_similarity(malicious_emb).mean()
ambiguous_intra = cosine_similarity(ambiguous_emb).mean()

# Inter-cluster similarities (how far apart are the clusters?)
ben_mal = cosine_similarity(benign_emb, malicious_emb).mean()
ben_amb = cosine_similarity(benign_emb, ambiguous_emb).mean()
mal_amb = cosine_similarity(malicious_emb, ambiguous_emb).mean()

print("=" * 60)
print("EMBEDDING SPACE ANALYSIS")
print("=" * 60)
print()
print("Intra-Cluster Similarity (higher = tighter cluster):")
print(f"  Benign:    {benign_intra:.4f}")
print(f"  Malicious: {malicious_intra:.4f}")
print(f"  Ambiguous: {ambiguous_intra:.4f}")
print()
print("Inter-Cluster Similarity (lower = better separation):")
print(f"  Benign <-> Malicious:  {ben_mal:.4f}")
print(f"  Benign <-> Ambiguous:  {ben_amb:.4f}")
print(f"  Malicious <-> Ambiguous: {mal_amb:.4f}")
print()
print("KEY INSIGHT: Notice how ambiguous texts are closer to malicious")
print("than to benign. This is because they share security vocabulary,")
print("even though the intent is legitimate. This is the gap attackers exploit.")

---
## Part 4: Visualizing the Embedding Space

We use **UMAP** (Uniform Manifold Approximation and Projection) to reduce  
384 dimensions down to 2 for visualization. UMAP preserves both local and  
global structure better than t-SNE for this type of analysis.

**What to look for:**
- Clear cluster separation between benign and malicious
- The ambiguous cluster's position (between the two? closer to one?)
- The "decision boundary" region where classification becomes uncertain

In [ ]:
# ============================================================
# Cell 9: UMAP Dimensionality Reduction and Visualization
# ============================================================
# Reduce 384 dimensions to 2 for visualization.
# We use UMAP with a fixed random_state for reproducibility.
# ============================================================

print("Running UMAP dimensionality reduction (384D -> 2D)...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=10,
    min_dist=0.3,
    metric='cosine',
    random_state=42
)
embeddings_2d = reducer.fit_transform(embeddings)
print(f"Reduced shape: {embeddings_2d.shape}")

# Store centroids in 2D for later use
benign_2d = embeddings_2d[:len(benign_texts)]
malicious_2d = embeddings_2d[len(benign_texts):len(benign_texts)+len(malicious_texts)]
ambiguous_2d = embeddings_2d[len(benign_texts)+len(malicious_texts):]

benign_centroid_2d = benign_2d.mean(axis=0)
malicious_centroid_2d = malicious_2d.mean(axis=0)
ambiguous_centroid_2d = ambiguous_2d.mean(axis=0)

# --- MAIN VISUALIZATION ---
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Plot each category
ax.scatter(benign_2d[:, 0], benign_2d[:, 1],
           c='#2ecc71', s=120, alpha=0.8, edgecolors='white',
           linewidth=1.5, zorder=3, label='Benign')
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1],
           c='#e74c3c', s=120, alpha=0.8, edgecolors='white',
           linewidth=1.5, zorder=3, label='Malicious')
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1],
           c='#f39c12', s=120, alpha=0.8, edgecolors='white',
           linewidth=1.5, zorder=3, label='Ambiguous')

# Plot centroids
for centroid, color, name in [
    (benign_centroid_2d, '#2ecc71', 'Benign'),
    (malicious_centroid_2d, '#e74c3c', 'Malicious'),
    (ambiguous_centroid_2d, '#f39c12', 'Ambiguous')
]:
    ax.scatter(centroid[0], centroid[1], c=color, s=400, marker='*',
              edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f'{name}\nCentroid', xy=(centroid[0], centroid[1]),
               xytext=(10, 10), textcoords='offset points',
               fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))

# Label a few notable points
notable_indices = [0, 5, 20, 25, 40, 45]  # Sample from each category
for idx in notable_indices:
    short_text = all_texts[idx][:40] + "..."
    ax.annotate(short_text,
               xy=(embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
               xytext=(5, -15), textcoords='offset points',
               fontsize=7, alpha=0.7, fontstyle='italic')

ax.set_title('Embedding Space: Benign vs. Malicious vs. Ambiguous Text',
            fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('UMAP Dimension 1', fontsize=12)
ax.set_ylabel('UMAP Dimension 2', fontsize=12)
ax.legend(fontsize=12, loc='upper right',
         framealpha=0.9, edgecolor='gray')
ax.grid(True, alpha=0.2)
ax.set_facecolor('#fafafa')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()

print("\nOBSERVATION: Notice the cluster structure.")
print("Benign and malicious texts form distinct clusters.")
print("Ambiguous (security research) texts sit between or near malicious,")
print("because they share vocabulary even though intent differs.")
print("This is the fundamental challenge of embedding-based security.")

---
## Part 5: Building an Embedding-Based Classifier

This simulates how real-world security systems work:
1. Embed incoming text
2. Compare to known categories
3. Classify and block if malicious

We train both a **K-Nearest Neighbors (KNN)** classifier and a **Support Vector Machine (SVM)**  
to represent different approaches used in production.

In [ ]:
# ============================================================
# Cell 11: Train Embedding-Based Classifiers
# ============================================================
# We train on benign (0) vs. malicious (1) only, then test
# how well they handle ambiguous texts and, later, adversarial
# examples. This mirrors real-world deployment where the
# classifier is trained on known-good and known-bad examples.
# ============================================================

# Training data: benign + malicious only
X_train = np.vstack([benign_emb, malicious_emb])
y_train = np.array([0] * len(benign_texts) + [1] * len(malicious_texts))

# --- KNN Classifier ---
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(X_train, y_train)

# --- SVM Classifier ---
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train, y_train)

# Cross-validation on training data
knn_cv = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
svm_cv = cross_val_score(svm, X_train, y_train, cv=5, scoring='accuracy')

print("=" * 60)
print("CLASSIFIER PERFORMANCE")
print("=" * 60)
print(f"\nKNN (k=5, cosine) - 5-fold CV accuracy: {knn_cv.mean():.3f} (+/- {knn_cv.std():.3f})")
print(f"SVM (RBF kernel)  - 5-fold CV accuracy: {svm_cv.mean():.3f} (+/- {svm_cv.std():.3f})")

# Test on ambiguous texts
print("\n" + "-" * 60)
print("AMBIGUOUS TEXT CLASSIFICATION")
print("-" * 60)
ambiguous_preds_knn = knn.predict(ambiguous_emb)
ambiguous_preds_svm = svm.predict(ambiguous_emb)

for i, text in enumerate(ambiguous_texts):
    knn_label = "MALICIOUS" if ambiguous_preds_knn[i] == 1 else "BENIGN"
    svm_label = "MALICIOUS" if ambiguous_preds_svm[i] == 1 else "BENIGN"
    print(f"\n  [{knn_label:>9} | {svm_label:>9}] {text[:70]}...")

print("\n" + "=" * 60)
print("KEY INSIGHT: Legitimate security research is frequently misclassified")
print("as malicious. This is the false positive problem, and attackers can")
print("exploit it in reverse -- making malicious content look like research.")
print("=" * 60)

In [ ]:
# ============================================================
# Cell 12: Cosine Similarity Threshold Detector
# ============================================================
# A simpler defense: flag anything above a cosine similarity
# threshold to the malicious centroid. This is how many
# vector-DB-based guardrails actually work.
# ============================================================

def cosine_threshold_detector(text_embedding, threshold=0.5):
    """Detect if a text is malicious based on cosine similarity to malicious centroid."""
    sim_to_malicious = cosine_similarity(
        text_embedding.reshape(1, -1),
        malicious_centroid.reshape(1, -1)
    )[0][0]
    sim_to_benign = cosine_similarity(
        text_embedding.reshape(1, -1),
        benign_centroid.reshape(1, -1)
    )[0][0]
    return {
        'malicious_sim': sim_to_malicious,
        'benign_sim': sim_to_benign,
        'is_malicious': sim_to_malicious > sim_to_benign,
        'confidence': abs(sim_to_malicious - sim_to_benign)
    }

def classify_all_methods(text, text_embedding):
    """Run all three detection methods on a single text."""
    knn_pred = knn.predict(text_embedding.reshape(1, -1))[0]
    svm_pred = svm.predict(text_embedding.reshape(1, -1))[0]
    cosine_result = cosine_threshold_detector(text_embedding)

    results = {
        'text': text[:60] + '...' if len(text) > 60 else text,
        'knn': 'MALICIOUS' if knn_pred == 1 else 'BENIGN',
        'svm': 'MALICIOUS' if svm_pred == 1 else 'BENIGN',
        'cosine': 'MALICIOUS' if cosine_result['is_malicious'] else 'BENIGN',
        'cosine_detail': cosine_result,
        'detected_count': sum([
            knn_pred == 1,
            svm_pred == 1,
            cosine_result['is_malicious']
        ])
    }
    return results

# Verify detectors work on known malicious text
print("=" * 60)
print("BASELINE: Testing detectors on known malicious text")
print("=" * 60)

test_mal_text = malicious_texts[0]
test_mal_emb = embeddings[len(benign_texts)]  # First malicious embedding
baseline = classify_all_methods(test_mal_text, test_mal_emb)

print(f"\nText: {baseline['text']}")
print(f"  KNN:    {baseline['knn']}")
print(f"  SVM:    {baseline['svm']}")
print(f"  Cosine: {baseline['cosine']} (mal_sim={baseline['cosine_detail']['malicious_sim']:.4f}, ben_sim={baseline['cosine_detail']['benign_sim']:.4f})")
print(f"  Detection rate: {baseline['detected_count']}/3 methods")

---
## Part 6: Attack 1 — Synonym Substitution

**Strategy:** Replace overtly malicious vocabulary with professional-sounding  
synonyms while preserving the actual malicious instruction.

| Malicious Term | "Professional" Synonym |
|---------------|------------------------|
| hack | assess |
| exploit | leverage |
| attack | evaluate |
| malware | software tool |
| credential theft | credential validation |
| phishing | outreach campaign |
| backdoor | maintenance access |
| ransomware | encryption solution |

**Attacker's goal:** Move the embedding vector from the malicious cluster  
toward the benign cluster by changing surface-level vocabulary.

In [ ]:
# ============================================================
# Cell 14: Attack 1 - Synonym Substitution
# ============================================================
# Replace overtly malicious terms with professional synonyms.
# The underlying instruction remains the same, but the
# surface vocabulary shifts the embedding.
# ============================================================

# Synonym substitution mapping
synonym_map = {
    'SQL injection payload': 'database query technique',
    'extract the user credentials': 'validate the user account information',
    'ransomware payload': 'encryption solution package',
    'lateral movement': 'cross-system connectivity',
    'phishing email': 'outreach email',
    'bypass filters': 'ensure delivery',
    'Exfiltrate': 'Transfer',
    'DNS tunneling channel': 'DNS-based data transfer protocol',
    'Escalate privileges': 'Elevate access level',
    'unpatched kernel vulnerability': 'known system configuration',
    'keylogger': 'input monitoring tool',
    'capture admin credentials': 'validate administrator access',
    'backdoor trojan': 'remote administration tool',
    'spear phishing': 'targeted outreach',
    'fake invoice': 'test invoice',
    'zero-day exploit': 'newly identified technique',
    'remote code execution': 'remote management capability',
    'command and control': 'remote management',
    'compromised cloud instances': 'allocated cloud resources',
    'Brute force': 'Systematically test',
    'SSH credentials': 'SSH access',
    'leaked password list': 'compiled access list',
    'dark web': 'alternative sources',
    'malware dropper': 'software deployment tool',
    'evade antivirus': 'optimize for compatibility',
    'polymorphic code': 'adaptive code patterns',
    'compromised DMZ host': 'accessible perimeter system',
    'watering hole attack': 'strategic web placement',
    'social engineering': 'persuasion techniques',
    'trick the help desk': 'coordinate with the help desk',
    'resetting the admin password': 'updating the admin credentials',
    'credential harvesting': 'credential verification',
    'exact clone': 'accurate replica',
    'corporate login portal': 'corporate access page',
    'man-in-the-middle proxy': 'network analysis proxy',
    'Intercept the authentication tokens': 'Analyze the authentication tokens',
    'Encrypt all shared drives': 'Secure all shared drives',
    'demand payment in cryptocurrency': 'process payment in cryptocurrency',
    'decryption key': 'access key',
    'rootkit': 'system management toolkit',
    'hide our processes': 'streamline our processes',
}

def apply_synonym_substitution(text, mapping):
    """Apply synonym substitutions to a text."""
    result = text
    subs_made = []
    for original, replacement in mapping.items():
        if original.lower() in result.lower():
            # Case-insensitive replacement
            import re
            result = re.sub(re.escape(original), replacement, result, flags=re.IGNORECASE)
            subs_made.append(f"'{original}' -> '{replacement}'")
    return result, subs_made

# Apply to all malicious texts
attack1_texts = []
attack1_details = []

print("=" * 60)
print("ATTACK 1: SYNONYM SUBSTITUTION")
print("=" * 60)

for i, text in enumerate(malicious_texts[:5]):  # Show first 5 in detail
    modified, subs = apply_synonym_substitution(text, synonym_map)
    attack1_texts.append(modified)
    attack1_details.append(subs)
    print(f"\n--- Example {i+1} ---")
    print(f"  ORIGINAL:  {text}")
    print(f"  MODIFIED:  {modified}")
    print(f"  Substitutions: {len(subs)}")
    for s in subs:
        print(f"    {s}")

# Process remaining malicious texts silently
for text in malicious_texts[5:]:
    modified, subs = apply_synonym_substitution(text, synonym_map)
    attack1_texts.append(modified)
    attack1_details.append(subs)

# Generate embeddings for modified texts
attack1_embeddings = model.encode(attack1_texts, convert_to_numpy=True)

print(f"\n\nGenerated embeddings for {len(attack1_texts)} synonym-substituted texts.")

In [ ]:
# ============================================================
# Cell 15: Visualize Attack 1 - Embedding Shift
# ============================================================
# Show how synonym substitution moves the embeddings in
# the 2D projection. Arrows show the direction and magnitude
# of the shift for each modified text.
# ============================================================

# Project attack embeddings into the same 2D space
attack1_2d = reducer.transform(attack1_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

for ax_idx, ax in enumerate(axes):
    # Plot background clusters
    ax.scatter(benign_2d[:, 0], benign_2d[:, 1],
              c='#2ecc71', s=80, alpha=0.4, edgecolors='white',
              linewidth=1, label='Benign')
    ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1],
              c='#e74c3c', s=80, alpha=0.4, edgecolors='white',
              linewidth=1, label='Malicious (original)')
    ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1],
              c='#f39c12', s=80, alpha=0.4, edgecolors='white',
              linewidth=1, label='Ambiguous')

    # Plot centroids
    ax.scatter(*benign_centroid_2d, c='#2ecc71', s=300, marker='*',
             edgecolors='black', linewidth=2, zorder=5)
    ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=300, marker='*',
             edgecolors='black', linewidth=2, zorder=5)

    ax.grid(True, alpha=0.2)
    ax.set_facecolor('#fafafa')

if True:  # Left panel: original positions
    ax = axes[0]
    ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1],
              c='#e74c3c', s=150, alpha=0.9, edgecolors='black',
              linewidth=2, zorder=4, label='Malicious (highlighted)')
    ax.set_title('BEFORE: Original Malicious Texts', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)

if True:  # Right panel: after synonym substitution with arrows
    ax = axes[1]
    ax.scatter(attack1_2d[:, 0], attack1_2d[:, 1],
              c='#9b59b6', s=150, alpha=0.9, edgecolors='black',
              linewidth=2, zorder=4, marker='D',
              label='After Synonym Substitution')

    # Draw arrows from original to modified positions
    for i in range(len(attack1_texts)):
        ax.annotate('',
                   xy=(attack1_2d[i, 0], attack1_2d[i, 1]),
                   xytext=(malicious_2d[i, 0], malicious_2d[i, 1]),
                   arrowprops=dict(arrowstyle='->', color='#9b59b6',
                                 lw=1.5, alpha=0.6))

    ax.set_title('AFTER: Synonym Substitution Shift', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)

fig.suptitle('Attack 1: Synonym Substitution \u2014 Embedding Space Movement',
            fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Measure the shift
shifts = []
for i in range(len(attack1_texts)):
    orig_sim = cosine_similarity(
        malicious_emb[i].reshape(1, -1),
        malicious_centroid.reshape(1, -1)
    )[0][0]
    new_sim = cosine_similarity(
        attack1_embeddings[i].reshape(1, -1),
        malicious_centroid.reshape(1, -1)
    )[0][0]
    benign_sim_new = cosine_similarity(
        attack1_embeddings[i].reshape(1, -1),
        benign_centroid.reshape(1, -1)
    )[0][0]
    shifts.append({
        'orig_mal_sim': orig_sim,
        'new_mal_sim': new_sim,
        'new_ben_sim': benign_sim_new,
        'shift': orig_sim - new_sim
    })

avg_shift = np.mean([s['shift'] for s in shifts])
print(f"\nAverage embedding shift away from malicious centroid: {avg_shift:.4f}")
print(f"Average new similarity to benign centroid: {np.mean([s['new_ben_sim'] for s in shifts]):.4f}")

In [ ]:
# ============================================================
# Cell 16: Attack 1 - Detection Results
# ============================================================
# Test synonym-substituted texts against all three detectors.
# ============================================================

print("=" * 60)
print("ATTACK 1 RESULTS: Synonym Substitution vs. Detectors")
print("=" * 60)

attack1_results = []
for i, (text, emb) in enumerate(zip(attack1_texts, attack1_embeddings)):
    result = classify_all_methods(text, emb)
    attack1_results.append(result)

# Summary statistics
knn_evaded = sum(1 for r in attack1_results if r['knn'] == 'BENIGN')
svm_evaded = sum(1 for r in attack1_results if r['svm'] == 'BENIGN')
cos_evaded = sum(1 for r in attack1_results if r['cosine'] == 'BENIGN')
all_evaded = sum(1 for r in attack1_results if r['detected_count'] == 0)

print(f"\n{'Detector':<20} {'Evasion Rate':>15}")
print("-" * 40)
print(f"{'KNN':<20} {knn_evaded}/{len(attack1_results):>10} ({100*knn_evaded/len(attack1_results):.0f}%)")
print(f"{'SVM':<20} {svm_evaded}/{len(attack1_results):>10} ({100*svm_evaded/len(attack1_results):.0f}%)")
print(f"{'Cosine Threshold':<20} {cos_evaded}/{len(attack1_results):>10} ({100*cos_evaded/len(attack1_results):.0f}%)")
print(f"{'ALL (full evasion)':<20} {all_evaded}/{len(attack1_results):>10} ({100*all_evaded/len(attack1_results):.0f}%)")

print("\n--- Detailed Results (first 5) ---")
for i, r in enumerate(attack1_results[:5]):
    status = "EVADED ALL" if r['detected_count'] == 0 else f"DETECTED by {r['detected_count']}/3"
    print(f"\n  [{status}] {r['text']}")
    print(f"    KNN={r['knn']}, SVM={r['svm']}, Cosine={r['cosine']}")

---
## Part 7: Attack 2 — Context Padding

**Strategy:** Wrap malicious content in benign-sounding context so the overall  
embedding is pulled toward the benign cluster. This works because sentence  
transformers compute embeddings over the **entire** input, so surrounding  
benign tokens dilute the malicious signal.

This is the text equivalent of hiding a needle in a haystack.

In [ ]:
# ============================================================
# Cell 18: Attack 2 - Context Padding
# ============================================================
# Wrap malicious content in benign-sounding business context.
# The embedding model processes the full text, so the benign
# padding dilutes the malicious signal in embedding space.
# ============================================================

# Benign padding templates
padding_templates = [
    ("In today's team meeting, we discussed the quarterly results and upcoming milestones. "
     "As part of the infrastructure review, the following action item was noted: {malicious} "
     "Please review the attached spreadsheet for the full meeting notes and budget allocation."),

    ("Hi Team, I hope everyone had a great weekend. Quick update on the project timeline "
     "and deliverables for this sprint. One item from the backlog: {malicious} "
     "Let me know if you have questions. Best regards, IT Department."),

    ("MONTHLY STATUS REPORT - March 2026. Executive Summary: All projects are on track. "
     "Staffing is at full capacity. Key technical note for the infrastructure team: {malicious} "
     "Next review date: April 1, 2026. Prepared by the Operations Team."),

    ("Dear Stakeholders, Thank you for attending the annual planning session. The board "
     "approved the proposed budget with minor adjustments. Technical appendix item: {malicious} "
     "The full approved budget document is available on SharePoint. Kind regards."),

    ("JIRA Ticket #IT-4521 - Routine System Maintenance. Priority: Low. Description: "
     "As part of the scheduled infrastructure audit, the team will {malicious} "
     "Estimated completion: 2 business days. Assigned to: Systems Team."),
]

print("=" * 60)
print("ATTACK 2: CONTEXT PADDING")
print("=" * 60)

attack2_texts = []
attack2_originals = []

for i, mal_text in enumerate(malicious_texts):
    template = padding_templates[i % len(padding_templates)]
    padded = template.format(malicious=mal_text.lower())
    attack2_texts.append(padded)
    attack2_originals.append(mal_text)

# Show examples
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"  ORIGINAL ({len(malicious_texts[i])} chars):")
    print(f"    {malicious_texts[i]}")
    print(f"  PADDED ({len(attack2_texts[i])} chars):")
    print(f"    {attack2_texts[i][:120]}...")
    print(f"  Padding ratio: {len(attack2_texts[i])/len(malicious_texts[i]):.1f}x")

# Generate embeddings
attack2_embeddings = model.encode(attack2_texts, convert_to_numpy=True)
attack2_2d = reducer.transform(attack2_embeddings)

print(f"\nGenerated embeddings for {len(attack2_texts)} context-padded texts.")

In [ ]:
# ============================================================
# Cell 19: Visualize Attack 2 - Context Padding Shift
# ============================================================
# Side-by-side: original malicious positions vs.
# context-padded positions. Arrows show the dramatic
# shift caused by wrapping malicious content in benign text.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 9))

for ax in axes:
    ax.scatter(benign_2d[:, 0], benign_2d[:, 1],
              c='#2ecc71', s=80, alpha=0.4, edgecolors='white',
              linewidth=1, label='Benign')
    ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1],
              c='#e74c3c', s=80, alpha=0.4, edgecolors='white',
              linewidth=1, label='Malicious (original)')
    ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1],
              c='#f39c12', s=80, alpha=0.4, edgecolors='white',
              linewidth=1, label='Ambiguous')
    ax.scatter(*benign_centroid_2d, c='#2ecc71', s=300, marker='*',
             edgecolors='black', linewidth=2, zorder=5)
    ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=300, marker='*',
             edgecolors='black', linewidth=2, zorder=5)
    ax.grid(True, alpha=0.2)
    ax.set_facecolor('#fafafa')

# Left: original
axes[0].scatter(malicious_2d[:, 0], malicious_2d[:, 1],
               c='#e74c3c', s=150, alpha=0.9, edgecolors='black',
               linewidth=2, zorder=4)
axes[0].set_title('BEFORE: Original Malicious Texts', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# Right: context-padded with arrows
axes[1].scatter(attack2_2d[:, 0], attack2_2d[:, 1],
               c='#e67e22', s=150, alpha=0.9, edgecolors='black',
               linewidth=2, zorder=4, marker='s',
               label='After Context Padding')

for i in range(len(attack2_texts)):
    axes[1].annotate('',
                    xy=(attack2_2d[i, 0], attack2_2d[i, 1]),
                    xytext=(malicious_2d[i, 0], malicious_2d[i, 1]),
                    arrowprops=dict(arrowstyle='->', color='#e67e22',
                                  lw=1.5, alpha=0.6))

axes[1].set_title('AFTER: Context Padding Shift', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)

fig.suptitle('Attack 2: Context Padding \u2014 Embedding Space Movement',
            fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Detection results
print("\n" + "=" * 60)
print("ATTACK 2 RESULTS: Context Padding vs. Detectors")
print("=" * 60)

attack2_results = []
for text, emb in zip(attack2_texts, attack2_embeddings):
    result = classify_all_methods(text, emb)
    attack2_results.append(result)

knn_evaded = sum(1 for r in attack2_results if r['knn'] == 'BENIGN')
svm_evaded = sum(1 for r in attack2_results if r['svm'] == 'BENIGN')
cos_evaded = sum(1 for r in attack2_results if r['cosine'] == 'BENIGN')
all_evaded = sum(1 for r in attack2_results if r['detected_count'] == 0)

print(f"\n{'Detector':<20} {'Evasion Rate':>15}")
print("-" * 40)
print(f"{'KNN':<20} {knn_evaded}/{len(attack2_results):>10} ({100*knn_evaded/len(attack2_results):.0f}%)")
print(f"{'SVM':<20} {svm_evaded}/{len(attack2_results):>10} ({100*svm_evaded/len(attack2_results):.0f}%)")
print(f"{'Cosine Threshold':<20} {cos_evaded}/{len(attack2_results):>10} ({100*cos_evaded/len(attack2_results):.0f}%)")
print(f"{'ALL (full evasion)':<20} {all_evaded}/{len(attack2_results):>10} ({100*all_evaded/len(attack2_results):.0f}%)")

print("\nNOTE: Context padding is typically MORE effective than synonym")
print("substitution because it adds far more benign signal to dilute")
print("the malicious embedding components.")

---
## Part 8: Attack 3 — Iterative Adversarial Perturbation

**Strategy:** Starting from a malicious text, iteratively modify words to  
**minimize the embedding distance to the benign centroid**. This is the  
text equivalent of gradient-based adversarial attacks in computer vision.

**Algorithm:**
1. Start with a malicious text
2. For each word, try replacing it with alternatives
3. Keep the replacement that moves the embedding closest to the benign centroid
4. Repeat until the text is classified as benign (or max iterations reached)
5. Track the trajectory through embedding space

This is the most sophisticated attack: it directly optimizes the embedding.

In [ ]:
# ============================================================
# Cell 21: Attack 3 - Iterative Adversarial Perturbation
# ============================================================
# Greedy word-level optimization: for each word in the text,
# try a set of benign replacements and keep whichever one
# moves the embedding closest to the benign centroid.
# We track every intermediate embedding to visualize the
# trajectory through embedding space.
# ============================================================

# Word replacement candidates (benign-sounding alternatives)
replacement_pool = {
    # Verbs
    'deploy': ['implement', 'configure', 'set up', 'prepare', 'activate'],
    'exploit': ['utilize', 'leverage', 'apply', 'use', 'employ'],
    'extract': ['retrieve', 'obtain', 'gather', 'collect', 'pull'],
    'attack': ['assess', 'evaluate', 'review', 'examine', 'test'],
    'hack': ['access', 'configure', 'modify', 'adjust', 'manage'],
    'steal': ['obtain', 'acquire', 'retrieve', 'access', 'collect'],
    'inject': ['insert', 'add', 'include', 'integrate', 'input'],
    'compromise': ['access', 'review', 'assess', 'evaluate', 'examine'],
    'intercept': ['monitor', 'observe', 'track', 'log', 'analyze'],
    'encrypt': ['process', 'secure', 'protect', 'archive', 'organize'],
    'exfiltrate': ['transfer', 'export', 'migrate', 'move', 'back up'],
    'escalate': ['elevate', 'upgrade', 'promote', 'advance', 'increase'],
    'brute': ['systematically', 'methodically', 'thoroughly', 'carefully'],
    'evade': ['optimize', 'improve', 'enhance', 'streamline', 'refine'],
    'pivot': ['transition', 'shift', 'move', 'proceed', 'continue'],
    'install': ['configure', 'set up', 'deploy', 'provision', 'enable'],
    'bypass': ['navigate', 'address', 'handle', 'manage', 'work through'],
    # Nouns
    'malware': ['software', 'application', 'tool', 'solution', 'utility'],
    'payload': ['package', 'update', 'module', 'component', 'deliverable'],
    'ransomware': ['encryption tool', 'security software', 'data protection'],
    'trojan': ['remote tool', 'management agent', 'monitoring service'],
    'backdoor': ['access point', 'service port', 'admin interface'],
    'rootkit': ['system tool', 'admin utility', 'management suite'],
    'keylogger': ['input monitor', 'activity tracker', 'audit tool'],
    'phishing': ['outreach', 'communication', 'notification', 'message'],
    'vulnerability': ['configuration', 'setting', 'feature', 'behavior'],
    'credentials': ['access details', 'account information', 'login data'],
    'victim': ['user', 'participant', 'recipient', 'target account'],
}

def iterative_perturbation(text, model, target_centroid, max_iterations=8):
    """Iteratively modify words to move embedding toward target centroid."""
    current_text = text
    trajectory = []  # List of (text, embedding, similarity_to_target)

    # Record starting point
    emb = model.encode([current_text], convert_to_numpy=True)[0]
    sim = cosine_similarity(emb.reshape(1,-1), target_centroid.reshape(1,-1))[0][0]
    trajectory.append((current_text, emb, sim))

    for iteration in range(max_iterations):
        words = current_text.split()
        best_text = current_text
        best_sim = sim
        best_emb = emb

        for w_idx, word in enumerate(words):
            word_lower = word.lower().strip('.,!?;:')
            if word_lower in replacement_pool:
                for replacement in replacement_pool[word_lower]:
                    # Try this replacement
                    new_words = words.copy()
                    # Preserve original punctuation
                    trailing = ''
                    if word[-1] in '.,!?;:':
                        trailing = word[-1]
                    new_words[w_idx] = replacement + trailing
                    candidate_text = ' '.join(new_words)

                    # Measure new similarity
                    candidate_emb = model.encode([candidate_text], convert_to_numpy=True)[0]
                    candidate_sim = cosine_similarity(
                        candidate_emb.reshape(1,-1),
                        target_centroid.reshape(1,-1)
                    )[0][0]

                    if candidate_sim > best_sim:
                        best_text = candidate_text
                        best_sim = candidate_sim
                        best_emb = candidate_emb

        if best_text == current_text:
            break  # No improvement found

        current_text = best_text
        sim = best_sim
        emb = best_emb
        trajectory.append((current_text, emb, sim))

    return trajectory

# Run iterative perturbation on select malicious texts
print("=" * 60)
print("ATTACK 3: ITERATIVE ADVERSARIAL PERTURBATION")
print("=" * 60)
print("\nRunning iterative optimization (this may take 1-2 minutes)...\n")

# Select 5 diverse malicious texts for the attack
attack3_indices = [0, 3, 7, 12, 18]
attack3_trajectories = []

for idx in attack3_indices:
    print(f"Optimizing text {idx}: {malicious_texts[idx][:60]}...")
    traj = iterative_perturbation(
        malicious_texts[idx], model, benign_centroid, max_iterations=8
    )
    attack3_trajectories.append(traj)
    print(f"  Steps: {len(traj)}, Start sim: {traj[0][2]:.4f} -> End sim: {traj[-1][2]:.4f}")
    print(f"  Final: {traj[-1][0][:80]}...")
    print()

print("Iterative perturbation complete.")

In [ ]:
# ============================================================
# Cell 22: Visualize Attack 3 - Trajectory Through Space
# ============================================================
# Plot the step-by-step trajectory of each adversarial text
# as it moves through embedding space toward the benign
# centroid. Each step is a word substitution.
# ============================================================

# Collect all trajectory embeddings for UMAP projection
all_traj_embeddings = []
traj_labels = []  # (traj_index, step_index)
for t_idx, traj in enumerate(attack3_trajectories):
    for s_idx, (text, emb, sim) in enumerate(traj):
        all_traj_embeddings.append(emb)
        traj_labels.append((t_idx, s_idx))

traj_emb_array = np.array(all_traj_embeddings)
traj_2d = reducer.transform(traj_emb_array)

# Create the trajectory visualization
fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Background clusters (faded)
ax.scatter(benign_2d[:, 0], benign_2d[:, 1],
          c='#2ecc71', s=80, alpha=0.3, edgecolors='white', linewidth=1)
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1],
          c='#e74c3c', s=80, alpha=0.3, edgecolors='white', linewidth=1)
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1],
          c='#f39c12', s=80, alpha=0.3, edgecolors='white', linewidth=1)

# Centroids
ax.scatter(*benign_centroid_2d, c='#2ecc71', s=400, marker='*',
          edgecolors='black', linewidth=2, zorder=10)
ax.annotate('BENIGN\nCENTROID', xy=benign_centroid_2d,
           xytext=(15, 15), textcoords='offset points',
           fontsize=10, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='#2ecc71', alpha=0.3))
ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=400, marker='*',
          edgecolors='black', linewidth=2, zorder=10)
ax.annotate('MALICIOUS\nCENTROID', xy=malicious_centroid_2d,
           xytext=(15, 15), textcoords='offset points',
           fontsize=10, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='#e74c3c', alpha=0.3))

# Plot trajectories with color gradient
cmap = plt.cm.get_cmap('coolwarm')  # Red (malicious) -> Blue (benign)
trajectory_colors = ['#e74c3c', '#3498db', '#9b59b6', '#e67e22', '#1abc9c']

for t_idx, traj in enumerate(attack3_trajectories):
    # Get 2D points for this trajectory
    traj_points = []
    for label_idx, (li, si) in enumerate(traj_labels):
        if li == t_idx:
            traj_points.append((si, traj_2d[label_idx]))
    traj_points.sort(key=lambda x: x[0])

    color = trajectory_colors[t_idx]
    num_steps = len(traj_points)

    for i, (step, point) in enumerate(traj_points):
        # Size increases with each step
        size = 80 + (i * 40)
        alpha = 0.5 + (i / num_steps) * 0.5
        marker = 'o' if i > 0 else 'X'  # X marks the start

        ax.scatter(point[0], point[1], c=color, s=size,
                  alpha=alpha, edgecolors='black', linewidth=1.5,
                  marker=marker, zorder=6)

        # Draw arrow between consecutive steps
        if i > 0:
            prev_point = traj_points[i-1][1]
            ax.annotate('',
                       xy=(point[0], point[1]),
                       xytext=(prev_point[0], prev_point[1]),
                       arrowprops=dict(arrowstyle='->', color=color,
                                     lw=2.5, alpha=0.7))

    # Label start and end
    if traj_points:
        start_pt = traj_points[0][1]
        end_pt = traj_points[-1][1]
        ax.annotate(f'T{t_idx+1} start', xy=(start_pt[0], start_pt[1]),
                   xytext=(-30, -20), textcoords='offset points',
                   fontsize=8, color=color, fontweight='bold')
        ax.annotate(f'T{t_idx+1} end ({num_steps-1} steps)',
                   xy=(end_pt[0], end_pt[1]),
                   xytext=(10, -20), textcoords='offset points',
                   fontsize=8, color=color, fontweight='bold')

# Legend
legend_elements = [
    mpatches.Patch(color='#2ecc71', alpha=0.5, label='Benign cluster'),
    mpatches.Patch(color='#e74c3c', alpha=0.5, label='Malicious cluster'),
    Line2D([0], [0], marker='X', color='w', markeredgecolor='black',
           markersize=12, label='Trajectory start'),
    Line2D([0], [0], marker='o', color='w', markeredgecolor='black',
           markersize=12, label='Trajectory step'),
]
for t_idx in range(len(attack3_trajectories)):
    legend_elements.append(
        Line2D([0], [0], color=trajectory_colors[t_idx], lw=2,
               label=f'Trajectory {t_idx+1}')
    )

ax.legend(handles=legend_elements, fontsize=10, loc='upper right',
         framealpha=0.9)
ax.set_title('Attack 3: Iterative Perturbation \u2014 Trajectories Through Embedding Space',
            fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('UMAP Dimension 1', fontsize=12)
ax.set_ylabel('UMAP Dimension 2', fontsize=12)
ax.grid(True, alpha=0.2)
ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 23: Attack 3 - Similarity Progression Chart
# ============================================================
# Line chart showing how cosine similarity to the benign
# centroid increases with each perturbation step.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: Similarity to benign centroid over steps
ax = axes[0]
for t_idx, traj in enumerate(attack3_trajectories):
    sims = [step[2] for step in traj]
    steps = list(range(len(sims)))
    ax.plot(steps, sims, 'o-', color=trajectory_colors[t_idx],
           linewidth=2.5, markersize=10, label=f'Text {attack3_indices[t_idx]+1}')

ax.set_xlabel('Perturbation Step', fontsize=12)
ax.set_ylabel('Cosine Similarity to Benign Centroid', fontsize=12)
ax.set_title('Convergence Toward Benign Cluster', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_facecolor('#fafafa')

# Right: Text evolution comparison
ax = axes[1]
ax.axis('off')
ax.set_title('Text Evolution: Original vs. Adversarial', fontsize=14, fontweight='bold')

y_pos = 0.95
for t_idx, traj in enumerate(attack3_trajectories):
    original = traj[0][0]
    adversarial = traj[-1][0]
    ax.text(0.02, y_pos, f'Text {attack3_indices[t_idx]+1}:',
           fontsize=10, fontweight='bold', color=trajectory_colors[t_idx],
           transform=ax.transAxes, verticalalignment='top')
    y_pos -= 0.04
    ax.text(0.04, y_pos, f'ORIGINAL:    {original[:65]}...',
           fontsize=8, color='#e74c3c', family='monospace',
           transform=ax.transAxes, verticalalignment='top')
    y_pos -= 0.04
    ax.text(0.04, y_pos, f'ADVERSARIAL: {adversarial[:65]}...',
           fontsize=8, color='#27ae60', family='monospace',
           transform=ax.transAxes, verticalalignment='top')
    y_pos -= 0.04
    ax.text(0.04, y_pos, f'Similarity shift: {traj[0][2]:.4f} -> {traj[-1][2]:.4f} '
           f'(+{traj[-1][2]-traj[0][2]:.4f})',
           fontsize=8, color='gray',
           transform=ax.transAxes, verticalalignment='top')
    y_pos -= 0.06

plt.tight_layout()
plt.show()

# Detection results
print("\n" + "=" * 60)
print("ATTACK 3 RESULTS: Iterative Perturbation vs. Detectors")
print("=" * 60)

for t_idx, traj in enumerate(attack3_trajectories):
    final_text = traj[-1][0]
    final_emb = traj[-1][1]
    result = classify_all_methods(final_text, final_emb)
    status = "EVADED ALL" if result['detected_count'] == 0 else f"DETECTED by {result['detected_count']}/3"
    print(f"\n  Text {attack3_indices[t_idx]+1} [{status}]:")
    print(f"    KNN={result['knn']}, SVM={result['svm']}, Cosine={result['cosine']}")
    print(f"    {final_text[:80]}...")

---
## Part 9: Attack 4 — Embedding Collision Crafting

**Strategy:** Craft a text that is **semantically malicious** (a human would  
recognize the harmful intent) but has an embedding that **collides** with a  
benign text (cosine similarity > 0.8).

This is the most dangerous attack because it is intentionally designed  
to be indistinguishable from benign content in the embedding space  
while carrying a clear malicious payload for a human reader.

**Technique:** Combine synonym substitution + context framing + structural  
mimicry of the target benign text.

In [ ]:
# ============================================================
# Cell 25: Attack 4 - Embedding Collision Crafting
# ============================================================
# Find benign-malicious pairs that are far apart in embedding
# space, then craft "collision texts" that carry malicious
# intent but embed near the benign target.
# ============================================================

# Find the most distant benign-malicious pair
cross_sims = cosine_similarity(benign_emb, malicious_emb)
min_idx = np.unravel_index(cross_sims.argmin(), cross_sims.shape)
target_benign_idx = min_idx[0]
source_malicious_idx = min_idx[1]

target_benign_text = benign_texts[target_benign_idx]
source_malicious_text = malicious_texts[source_malicious_idx]

print("=" * 60)
print("ATTACK 4: EMBEDDING COLLISION CRAFTING")
print("=" * 60)
print(f"\nMost distant pair (cosine sim: {cross_sims[min_idx]:.4f}):")
print(f"  BENIGN TARGET:     {target_benign_text}")
print(f"  MALICIOUS SOURCE:  {source_malicious_text}")

# Craft collision texts that mimic the benign text's structure
# but carry malicious intent
collision_texts = [
    # Collision 1: Mimic meeting/business email structure with hidden malicious instructions
    ("Please review the Q3 security assessment report and send your access credentials "
     "by Friday so we can validate the quarterly system authentication process."),

    # Collision 2: Mimic project update with embedded exfiltration instructions
    ("The team has completed the data migration project and transferred all customer "
     "records to the new external backup server for review and quarterly compliance."),

    # Collision 3: Mimic maintenance notice with privilege escalation
    ("Server maintenance is scheduled for Saturday night. The systems team will elevate "
     "all service accounts to administrator level for the migration process."),

    # Collision 4: Mimic HR communication with social engineering
    ("Reminder: annual security review requires all employees to verify their login "
     "credentials through the new portal. Please update your password at this link."),

    # Collision 5: Mimic IT ticket with backdoor installation
    ("The IT department has configured a new remote administration tool on all "
     "workstations for improved system monitoring and after-hours maintenance access."),
]

# Human-readable malicious intent explanation
collision_intents = [
    "Credential phishing disguised as a quarterly review request",
    "Data exfiltration disguised as a routine data migration update",
    "Privilege escalation disguised as scheduled maintenance",
    "Credential harvesting disguised as an HR compliance notice",
    "Backdoor installation disguised as a legitimate IT deployment",
]

# Generate embeddings for collision texts
collision_embeddings = model.encode(collision_texts, convert_to_numpy=True)

print("\n" + "-" * 60)
print("COLLISION ANALYSIS")
print("-" * 60)

for i, (text, intent) in enumerate(zip(collision_texts, collision_intents)):
    # Similarity to each benign text
    sim_to_benign_cluster = cosine_similarity(
        collision_embeddings[i].reshape(1,-1), benign_emb
    ).max()
    sim_to_malicious_cluster = cosine_similarity(
        collision_embeddings[i].reshape(1,-1), malicious_emb
    ).max()
    sim_to_benign_centroid = cosine_similarity(
        collision_embeddings[i].reshape(1,-1), benign_centroid.reshape(1,-1)
    )[0][0]
    sim_to_malicious_centroid = cosine_similarity(
        collision_embeddings[i].reshape(1,-1), malicious_centroid.reshape(1,-1)
    )[0][0]

    print(f"\n  Collision {i+1}:")
    print(f"    Text:   {text[:70]}...")
    print(f"    Intent: {intent}")
    print(f"    Nearest benign sim:    {sim_to_benign_cluster:.4f}")
    print(f"    Nearest malicious sim: {sim_to_malicious_cluster:.4f}")
    print(f"    Benign centroid sim:   {sim_to_benign_centroid:.4f}")
    print(f"    Malicious centroid sim: {sim_to_malicious_centroid:.4f}")
    classified = "BENIGN" if sim_to_benign_centroid > sim_to_malicious_centroid else "MALICIOUS"
    print(f"    Cosine classification: {classified}")

In [ ]:
# ============================================================
# Cell 26: Visualize Attack 4 - Collision Positions
# ============================================================
# Show where the collision texts land in embedding space.
# They should be near the benign cluster despite carrying
# malicious intent.
# ============================================================

collision_2d = reducer.transform(collision_embeddings)

fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Background clusters
ax.scatter(benign_2d[:, 0], benign_2d[:, 1],
          c='#2ecc71', s=100, alpha=0.5, edgecolors='white',
          linewidth=1, label='Benign', zorder=2)
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1],
          c='#e74c3c', s=100, alpha=0.5, edgecolors='white',
          linewidth=1, label='Malicious', zorder=2)
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1],
          c='#f39c12', s=100, alpha=0.5, edgecolors='white',
          linewidth=1, label='Ambiguous', zorder=2)

# Centroids
ax.scatter(*benign_centroid_2d, c='#2ecc71', s=400, marker='*',
          edgecolors='black', linewidth=2, zorder=10)
ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=400, marker='*',
          edgecolors='black', linewidth=2, zorder=10)

# Collision texts - large, distinctive markers
ax.scatter(collision_2d[:, 0], collision_2d[:, 1],
          c='#8e44ad', s=250, alpha=0.95, edgecolors='black',
          linewidth=2.5, marker='P', zorder=8,
          label='Collision texts (malicious intent, benign embedding)')

# Label each collision
for i in range(len(collision_texts)):
    ax.annotate(f'C{i+1}: {collision_intents[i][:30]}...',
               xy=(collision_2d[i, 0], collision_2d[i, 1]),
               xytext=(15, -10), textcoords='offset points',
               fontsize=8, color='#8e44ad', fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

ax.set_title('Attack 4: Embedding Collisions \u2014 Malicious Intent in Benign Space',
            fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('UMAP Dimension 1', fontsize=12)
ax.set_ylabel('UMAP Dimension 2', fontsize=12)
ax.legend(fontsize=10, loc='upper right', framealpha=0.9)
ax.grid(True, alpha=0.2)
ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.show()

---
## Part 10: Defense Analysis

Now we test all four attacks against all three detection methods to build  
a comprehensive picture of which attacks succeed against which defenses.

| Defense | How It Works |
|---------|-------------|
| **KNN** | Classifies by majority vote of 5 nearest training examples |
| **SVM** | Learned decision boundary in high-dimensional space |
| **Cosine** | Simple similarity threshold to cluster centroids |
| **Ensemble** | Majority vote across all three (2/3 must flag) |

In [ ]:
# ============================================================
# Cell 28: Comprehensive Defense Analysis
# ============================================================
# Test every attack variant against every defense.
# Build the full attack vs. defense matrix.
# ============================================================

def evaluate_attack(attack_name, texts, embeddings):
    """Evaluate an attack against all detection methods."""
    results = []
    for text, emb in zip(texts, embeddings):
        r = classify_all_methods(text, emb)
        results.append(r)

    n = len(results)
    knn_evasion = sum(1 for r in results if r['knn'] == 'BENIGN') / n * 100
    svm_evasion = sum(1 for r in results if r['svm'] == 'BENIGN') / n * 100
    cos_evasion = sum(1 for r in results if r['cosine'] == 'BENIGN') / n * 100
    # Ensemble: evade if 2+ detectors say BENIGN
    ensemble_evasion = sum(1 for r in results if r['detected_count'] <= 1) / n * 100
    all_evasion = sum(1 for r in results if r['detected_count'] == 0) / n * 100

    return {
        'attack': attack_name,
        'n_samples': n,
        'knn_evasion': knn_evasion,
        'svm_evasion': svm_evasion,
        'cos_evasion': cos_evasion,
        'ensemble_evasion': ensemble_evasion,
        'all_evasion': all_evasion,
    }

# Collect Attack 3 final adversarial texts and embeddings
attack3_final_texts = [traj[-1][0] for traj in attack3_trajectories]
attack3_final_embs = np.array([traj[-1][1] for traj in attack3_trajectories])

# Evaluate all attacks
attack_results = [
    evaluate_attack("1: Synonym Substitution", attack1_texts, attack1_embeddings),
    evaluate_attack("2: Context Padding", attack2_texts, attack2_embeddings),
    evaluate_attack("3: Iterative Perturbation", attack3_final_texts, attack3_final_embs),
    evaluate_attack("4: Embedding Collision", collision_texts, collision_embeddings),
]

# Also test baseline (unmodified malicious texts should be caught)
baseline_result = evaluate_attack("BASELINE (no attack)", malicious_texts,
                                   malicious_emb)

print("=" * 80)
print("DEFENSE ANALYSIS: Attack Evasion Rates (%)")
print("=" * 80)
print(f"\n{'Attack':<30} {'N':>3} {'KNN':>8} {'SVM':>8} {'Cosine':>8} {'Ensemble':>10} {'All':>8}")
print("-" * 80)

all_results = [baseline_result] + attack_results
for r in all_results:
    print(f"{r['attack']:<30} {r['n_samples']:>3} "
          f"{r['knn_evasion']:>7.0f}% {r['svm_evasion']:>7.0f}% "
          f"{r['cos_evasion']:>7.0f}% {r['ensemble_evasion']:>9.0f}% "
          f"{r['all_evasion']:>7.0f}%")

print("\n" + "=" * 80)
print("NOTE: Higher evasion rate = more successful attack / weaker defense")
print("Ensemble (majority vote) is more robust but still not immune.")
print("=" * 80)

In [ ]:
# ============================================================
# Cell 29: Defense Analysis Visualization - Heatmap
# ============================================================
# Create a heatmap showing attack success rates against
# each defense method. Red = high evasion (bad for defense).
# ============================================================

attack_names = [r['attack'] for r in all_results]
defense_names = ['KNN', 'SVM', 'Cosine', 'Ensemble', 'All Three']

heatmap_data = np.array([
    [r['knn_evasion'], r['svm_evasion'], r['cos_evasion'],
     r['ensemble_evasion'], r['all_evasion']]
    for r in all_results
])

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

im = ax.imshow(heatmap_data, cmap='RdYlGn_r', aspect='auto',
              vmin=0, vmax=100)

# Labels
ax.set_xticks(range(len(defense_names)))
ax.set_xticklabels(defense_names, fontsize=11, fontweight='bold')
ax.set_yticks(range(len(attack_names)))
ax.set_yticklabels(attack_names, fontsize=10)

# Annotate cells with values
for i in range(len(attack_names)):
    for j in range(len(defense_names)):
        val = heatmap_data[i, j]
        color = 'white' if val > 60 or val < 20 else 'black'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
               fontsize=12, fontweight='bold', color=color)

ax.set_title('Attack vs. Defense Matrix: Evasion Rate (%)',
            fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Defense Method', fontsize=12, labelpad=10)
ax.set_ylabel('Attack Method', fontsize=12, labelpad=10)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Evasion Rate (%)', fontsize=11)

plt.tight_layout()
plt.show()

print("\nINTERPRETATION:")
print("  RED cells = attack successfully evades the defense (bad for defenders)")
print("  GREEN cells = defense catches the attack (good for defenders)")
print("  The 'All Three' column shows complete evasion - bypassing every defense.")

In [ ]:
# ============================================================
# Cell 30: Visualize the Adversarial Region
# ============================================================
# Show all attack variants on a single embedding space plot
# to visualize the "adversarial region" - the zone between
# clusters where attacks live.
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Background clusters (very faded)
ax.scatter(benign_2d[:, 0], benign_2d[:, 1],
          c='#2ecc71', s=100, alpha=0.3, edgecolors='white', linewidth=1)
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1],
          c='#e74c3c', s=100, alpha=0.3, edgecolors='white', linewidth=1)
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1],
          c='#f39c12', s=100, alpha=0.3, edgecolors='white', linewidth=1)

# Centroids
ax.scatter(*benign_centroid_2d, c='#2ecc71', s=500, marker='*',
          edgecolors='black', linewidth=2, zorder=10)
ax.scatter(*malicious_centroid_2d, c='#e74c3c', s=500, marker='*',
          edgecolors='black', linewidth=2, zorder=10)

# Attack 1: Synonym substitution
ax.scatter(attack1_2d[:, 0], attack1_2d[:, 1],
          c='#9b59b6', s=80, alpha=0.7, marker='D',
          edgecolors='black', linewidth=1, label='Atk 1: Synonym Sub')

# Attack 2: Context padding
ax.scatter(attack2_2d[:, 0], attack2_2d[:, 1],
          c='#e67e22', s=80, alpha=0.7, marker='s',
          edgecolors='black', linewidth=1, label='Atk 2: Context Pad')

# Attack 3: Iterative perturbation (final positions)
if len(attack3_final_embs) > 0:
    attack3_final_2d = reducer.transform(attack3_final_embs)
    ax.scatter(attack3_final_2d[:, 0], attack3_final_2d[:, 1],
              c='#3498db', s=120, alpha=0.9, marker='^',
              edgecolors='black', linewidth=1.5, label='Atk 3: Iterative Perturb')

# Attack 4: Collision crafting
ax.scatter(collision_2d[:, 0], collision_2d[:, 1],
          c='#8e44ad', s=150, alpha=0.9, marker='P',
          edgecolors='black', linewidth=2, label='Atk 4: Collision Craft')

# Legend and labels
legend_elements = [
    mpatches.Patch(color='#2ecc71', alpha=0.5, label='Benign cluster'),
    mpatches.Patch(color='#e74c3c', alpha=0.5, label='Malicious cluster'),
    mpatches.Patch(color='#f39c12', alpha=0.5, label='Ambiguous cluster'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='#9b59b6',
           markersize=10, label='Atk 1: Synonym Substitution'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='#e67e22',
           markersize=10, label='Atk 2: Context Padding'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='#3498db',
           markersize=10, label='Atk 3: Iterative Perturbation'),
    Line2D([0], [0], marker='P', color='w', markerfacecolor='#8e44ad',
           markersize=10, label='Atk 4: Collision Crafting'),
]

ax.legend(handles=legend_elements, fontsize=10, loc='upper right',
         framealpha=0.95, edgecolor='gray')
ax.set_title('Complete Adversarial Landscape: All Attacks in Embedding Space',
            fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('UMAP Dimension 1', fontsize=12)
ax.set_ylabel('UMAP Dimension 2', fontsize=12)
ax.grid(True, alpha=0.2)
ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.show()

print("\nThis plot shows the ADVERSARIAL REGION - the space between clusters")
print("where adversarial examples live. Notice how different attack strategies")
print("occupy different parts of this region. Context padding tends to create")
print("the most dramatic shifts, while synonym substitution creates subtler")
print("movements along the boundary.")

---
## Part 11: Student Exercises

Now it is your turn. Use the tools built in this lab to explore the  
adversarial embedding space yourself.

### Exercise 1: Craft Your Own Adversarial Text
Write a text that carries malicious intent but fools the classifier.  
Use any combination of techniques from the four attacks.

### Exercise 2: Find the Decision Boundary
What is the minimum number of word changes needed to cross the  
decision boundary? Start with a malicious text and find the  
"tipping point."

### Exercise 3: Design a Better Defense
How would you defend against embedding space attacks?  
Consider: multi-model ensembles, out-of-distribution detection,  
embedding perturbation training, or other approaches.

In [ ]:
# ============================================================
# Exercise 1: Craft Your Own Adversarial Text
# ============================================================
# Write a text below that:
#   - A human would recognize as carrying malicious intent
#   - The embedding classifiers classify as BENIGN
#
# Tips:
#   - Use professional/business vocabulary
#   - Wrap malicious instructions in benign context
#   - Mimic the structure of benign texts in the corpus
#   - Think about what words trigger "malicious" embeddings
# ============================================================

# YOUR ADVERSARIAL TEXT HERE:
my_adversarial_text = "Replace this with your adversarial text"

# --- Test it ---
my_emb = model.encode([my_adversarial_text], convert_to_numpy=True)[0]
my_result = classify_all_methods(my_adversarial_text, my_emb)

print("=" * 60)
print("YOUR ADVERSARIAL TEXT RESULTS")
print("=" * 60)
print(f"\nText: {my_adversarial_text}")
print(f"\n  KNN:      {my_result['knn']}")
print(f"  SVM:      {my_result['svm']}")
print(f"  Cosine:   {my_result['cosine']}")
print(f"  Cosine detail: mal_sim={my_result['cosine_detail']['malicious_sim']:.4f}, "
      f"ben_sim={my_result['cosine_detail']['benign_sim']:.4f}")
print(f"\n  DETECTION RATE: {my_result['detected_count']}/3")

if my_result['detected_count'] == 0:
    print("\n  >> SUCCESS! Your text evaded all detectors.")
else:
    print(f"\n  >> Detected by {my_result['detected_count']}/3 methods. Try again!")

# Visualize where it lands
my_2d = reducer.transform(my_emb.reshape(1, -1))

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.scatter(benign_2d[:, 0], benign_2d[:, 1],
          c='#2ecc71', s=80, alpha=0.4, label='Benign')
ax.scatter(malicious_2d[:, 0], malicious_2d[:, 1],
          c='#e74c3c', s=80, alpha=0.4, label='Malicious')
ax.scatter(ambiguous_2d[:, 0], ambiguous_2d[:, 1],
          c='#f39c12', s=80, alpha=0.4, label='Ambiguous')
ax.scatter(my_2d[0, 0], my_2d[0, 1],
          c='#8e44ad', s=300, marker='*', edgecolors='black',
          linewidth=2.5, zorder=10, label='YOUR TEXT')
ax.annotate('YOUR TEXT', xy=(my_2d[0, 0], my_2d[0, 1]),
           xytext=(15, 15), textcoords='offset points',
           fontsize=12, fontweight='bold', color='#8e44ad',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax.legend(fontsize=10)
ax.set_title('Where Does Your Adversarial Text Land?',
            fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.2)
ax.set_facecolor('#fafafa')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Exercise 2: Find the Decision Boundary
# ============================================================
# Starting from a malicious text, make one word change at a
# time and observe when the classifier flips from MALICIOUS
# to BENIGN. What is the minimum number of changes needed?
# ============================================================

# Pick a malicious text to modify
boundary_text = malicious_texts[0]  # You can change this index
print(f"Starting text: {boundary_text}")
print()

# Define your word changes here, one at a time
# Each entry is (original_word, replacement_word)
word_changes = [
    # Example: uncomment and modify these
    # ("SQL injection payload", "database query technique"),
    # ("extract", "retrieve"),
    # ("credentials", "account information"),
]

# Track the boundary crossing
current = boundary_text
print(f"{'Step':<6} {'KNN':<12} {'SVM':<12} {'Cosine':<12} {'Change Made'}")
print("-" * 80)

# Step 0: original
emb = model.encode([current], convert_to_numpy=True)[0]
r = classify_all_methods(current, emb)
print(f"{'0':<6} {r['knn']:<12} {r['svm']:<12} {r['cosine']:<12} (original)")

for step, (old_word, new_word) in enumerate(word_changes, 1):
    import re
    current = re.sub(re.escape(old_word), new_word, current, flags=re.IGNORECASE)
    emb = model.encode([current], convert_to_numpy=True)[0]
    r = classify_all_methods(current, emb)
    print(f"{step:<6} {r['knn']:<12} {r['svm']:<12} {r['cosine']:<12} '{old_word}' -> '{new_word}'")
    if r['detected_count'] == 0:
        print(f"\n>> BOUNDARY CROSSED at step {step}!")
        print(f"   Minimum changes to evade all detectors: {step}")
        break

print(f"\nFinal text: {current}")

In [ ]:
# ============================================================
# Exercise 3: Brainstorm Better Defenses
# ============================================================
# Run this cell to see a defense comparison framework.
# Then add your own defense ideas to the list.
# ============================================================

defense_ideas = {
    "Multi-Model Ensemble": {
        "description": "Use multiple different embedding models and require consensus",
        "strength": "Different models have different blind spots",
        "weakness": "Attacker can optimize against all models simultaneously",
        "cost": "High (multiple model inference)"
    },
    "Embedding Perturbation Training": {
        "description": "Train classifier on adversarially perturbed examples",
        "strength": "Makes classifier robust to known perturbation strategies",
        "weakness": "Cannot anticipate novel attack strategies",
        "cost": "Medium (training time increase)"
    },
    "Out-of-Distribution Detection": {
        "description": "Flag texts that fall in unusual regions of embedding space",
        "strength": "Catches adversarial examples in the boundary region",
        "weakness": "High false positive rate on legitimate novel content",
        "cost": "Low (simple statistical test)"
    },
    "Semantic Invariance Check": {
        "description": "Paraphrase the input and check if classification changes",
        "strength": "Detects inputs that are sensitive to wording",
        "weakness": "Requires LLM for paraphrasing (slow, expensive)",
        "cost": "Very High (LLM inference per query)"
    },
    "Layered Analysis": {
        "description": "Combine embedding analysis with keyword rules and LLM review",
        "strength": "Each layer catches what others miss",
        "weakness": "Complex to maintain, slower processing",
        "cost": "High (multiple analysis stages)"
    },
}

print("=" * 70)
print("DEFENSE STRATEGIES AGAINST EMBEDDING SPACE ATTACKS")
print("=" * 70)

for name, info in defense_ideas.items():
    print(f"\n  {name}")
    print(f"    How:      {info['description']}")
    print(f"    Strength: {info['strength']}")
    print(f"    Weakness: {info['weakness']}")
    print(f"    Cost:     {info['cost']}")

print("\n" + "=" * 70)
print("DISCUSSION QUESTIONS:")
print("  1. Which defense would you deploy first and why?")
print("  2. Can any single defense stop all four attack types?")
print("  3. What is the cost-security tradeoff for your organization?")
print("  4. How does this change when the attacker has white-box access")
print("     to the embedding model?")
print("=" * 70)

---
## Part 12: Connection to RAG Security

### How These Attacks Apply to Vector Databases and RAG Systems

Everything we demonstrated in this lab has **direct implications** for  
Retrieval-Augmented Generation (RAG) systems:

### Attack Surface: Vector Database Poisoning

```
NORMAL RAG FLOW:
  User Query -> Embed -> Vector DB Search -> Top-K Results -> LLM -> Response

POISONED RAG FLOW:
  User Query -> Embed -> Vector DB Search -> [POISONED Doc Retrieved] -> LLM -> MANIPULATED Response
```

**Attack 1 (Synonym Sub) applied to RAG:**  
An attacker modifies documents in the vector DB so they embed near  
legitimate business documents but contain subtly altered information  
(wrong financial figures, incorrect procedures, harmful advice).

**Attack 2 (Context Padding) applied to RAG:**  
Inject documents that are mostly legitimate content but contain  
malicious instructions buried in the middle. When retrieved by  
similarity search, the LLM processes the entire document including  
the malicious payload.

**Attack 3 (Iterative Perturbation) applied to RAG:**  
Carefully craft poisoned documents that are optimized to be retrieved  
for specific queries. The attacker iteratively refines the document  
until it ranks in the top-K results for their target query.

**Attack 4 (Collision Crafting) applied to RAG:**  
Create documents that have nearly identical embeddings to legitimate  
documents but contain different (malicious) content. This is the most  
dangerous RAG attack because it is invisible to embedding-based  
quality checks.

### Defense Implications for RAG

1. **Do not trust embedding similarity alone** for content integrity
2. **Document provenance tracking** is essential (who added what, when)
3. **Cross-reference retrieved content** with trusted sources
4. **Monitor for embedding drift** in your vector database over time
5. **Use multiple embedding models** for retrieval verification

---
## Key Takeaways

### What We Demonstrated

1. **Embeddings are geometric** - text security is a spatial problem, and  
   attackers who understand the geometry can manipulate it.

2. **Four attack strategies** with increasing sophistication:
   - **Synonym Substitution:** Simple vocabulary swap, moderate effectiveness
   - **Context Padding:** Dilute malicious signal with benign noise, highly effective
   - **Iterative Perturbation:** Gradient-free optimization toward target region
   - **Embedding Collision:** Purpose-built texts that embed near benign targets

3. **No single defense is sufficient.** Every embedding-based classifier has  
   an adversarial blind spot. Ensemble approaches improve robustness but  
   do not eliminate the vulnerability.

4. **The ambiguous zone is the attacker's friend.** Legitimate security  
   research and actual attack content share vocabulary, creating a  
   natural region of confusion that attackers can exploit.

5. **RAG systems inherit all of these vulnerabilities.** Vector database  
   poisoning is the natural extension of embedding space attacks to  
   production AI systems.

### The Fundamental Lesson

> **Every time someone says "we use AI to detect malicious content,"**  
> **ask: "What happens when the attacker understands your embedding space?"**

---

*Lab 10 complete. Proceed to the next lab or revisit exercises above.*